<a href="https://colab.research.google.com/github/umair594/Machine-Learning-Projects/blob/main/Predict_Book_Popularity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project 10: What Makes a Good Book?**

# **Predict Book Popularity Using Machine Learning**

**In this project, you will build a Book Popularity Prediction System that predicts whether a book can become popular based on:**

Book title

Description/Summary

Ratings

Review counts

Author-related information

Other numerical and textual features

**This is a real-world NLP + Machine Learning project because it combines:**

Text data → titles & descriptions

Numerical data → ratings, review counts

Feature engineering

Machine learning pipelines

Model optimization

# **Objective of the Project**

**The goal is to predict:**

“**Will a book become popular or not?**”

using machine learning techniques.

**This type of system is useful for:**

Online bookstores

Recommendation systems

Publishers

Marketing teams

Readers discovering trending books

# **Step 1 : Import Libraries**

In [1]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# **Step 2 : Load Dataset**

In [2]:
df = pd.read_csv("Books_Data_Clean.csv")

# **Step 3 : Quick EDA**

In [3]:
print(df.head())

   Publishing Year                        Book Name  \
0           1975.0                          Beowulf   
1           1987.0                 Batman: Year One   
2           2015.0                Go Set a Watchman   
3           2008.0  When You Are Engulfed in Flames   
4           2011.0         Daughter of Smoke & Bone   

                                              Author language_code  \
0                             Unknown, Seamus Heaney         en-US   
1  Frank Miller, David Mazzucchelli, Richmond Lew...           eng   
2                                         Harper Lee           eng   
3                                      David Sedaris         en-US   
4                                       Laini Taylor           eng   

  Author_Rating  Book_average_rating  Book_ratings_count          genre  \
0        Novice                 3.42              155903  genre fiction   
1  Intermediate                 4.23              145267  genre fiction   
2        Novice        

In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1070 entries, 0 to 1069
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Publishing Year      1069 non-null   float64
 1   Book Name            1050 non-null   object 
 2   Author               1070 non-null   object 
 3   language_code        1017 non-null   object 
 4   Author_Rating        1070 non-null   object 
 5   Book_average_rating  1070 non-null   float64
 6   Book_ratings_count   1070 non-null   int64  
 7   genre                1070 non-null   object 
 8   gross sales          1070 non-null   float64
 9   publisher revenue    1070 non-null   float64
 10  sale price           1070 non-null   float64
 11  sales rank           1070 non-null   int64  
 12  Publisher            1070 non-null   object 
 13  units sold           1070 non-null   int64  
dtypes: float64(5), int64(3), object(6)
memory usage: 117.2+ KB
None


In [5]:
print(df.isnull().sum())

Publishing Year         1
Book Name              20
Author                  0
language_code          53
Author_Rating           0
Book_average_rating     0
Book_ratings_count      0
genre                   0
gross sales             0
publisher revenue       0
sale price              0
sales rank              0
Publisher               0
units sold              0
dtype: int64


# **Step 4 : Understand the Dataset**

**Typical columns may include**:

| Column       | Description       |
| ------------ | ----------------- |
| Title        | Book name         |
| description  | Book summary      |
| ratingsCount | Number of ratings |
| reviewCount  | Number of reviews |
| rating       | Average rating    |
| genre        | Book category     |


# **Step 5 : Create Target Variable**

Suppose we define:

In [9]:
df['Popular'] = np.where(df['Book_ratings_count'] > 1000, 1, 0)

**Meaning:**

1 → Popular Book

0 → Not Popular

# **Step 6 : Handle Missing Values**

In [16]:
df['Book Name'] = df['Book Name'].fillna("")
df['language_code'] = df['language_code'].fillna("")
df['Publishing Year'] = df['Publishing Year'].fillna("")

In [17]:
df.isnull().sum()

,0
Publishing Year,0
Book Name,0
Author,0
language_code,0
Author_Rating,0
Book_average_rating,0
Book_ratings_count,0
genre,0
gross sales,0
publisher revenue,0


# **Step 7 : Select Features**

**Text Features**

Title

Description

**Numeric Features**

Rating

Review Count

In [19]:
X = df[['Book Name', 'Book_average_rating', 'Book_ratings_count']]
y = df['Popular']

# **Step 8 : Build NLP + Numeric Pipeline**

In [20]:
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin

class TextSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.key]

# **Step 9 : TF-IDF Feature Extraction**

In [28]:
title_pipeline = Pipeline([
    ('selector', TextSelector('Book Name')),
    ('tfidf', TfidfVectorizer(stop_words='english'))
])

# description_pipeline is removed as 'description' column does not exist in X

# **Step 10 : Numeric Pipeline**

In [22]:
class NumberSelector(BaseEstimator, TransformerMixin):
    def __init__(self, key):
        self.key = key

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[[self.key]]

In [29]:
rating_pipeline = Pipeline([
    ('selector', NumberSelector('Book_average_rating')),
    ('scaler', StandardScaler())
])

review_pipeline = Pipeline([
    ('selector', NumberSelector('Book_ratings_count')),
    ('scaler', StandardScaler())
])

# **Step 11 : Combine Features**

In [31]:
from sklearn.pipeline import FeatureUnion

features = FeatureUnion([
    ('title', title_pipeline),
    ('rating', rating_pipeline),
    ('reviews', review_pipeline)
])

# **Step 12 : Train-Test Split**

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# **Step 13 : Build Final ML Pipeline**

In [33]:
model = Pipeline([
    ('features', features),
    ('classifier', RandomForestClassifier())
])

# **Step 14 : Train Model**

In [34]:
model.fit(X_train, y_train)

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('title',
                                                 Pipeline(steps=[('selector',
                                                                  TextSelector(key='Book '
                                                                                   'Name')),
                                                                 ('tfidf',
                                                                  TfidfVectorizer(stop_words='english'))])),
                                                ('rating',
                                                 Pipeline(steps=[('selector',
                                                                  NumberSelector(key='Book_average_rating')),
                                                                 ('scaler',
                                                                  StandardScaler())])),
                                                ('reviews',
                                                 Pipeline(steps=[('selector',
                                                                  NumberSelector(key='Book_ratings_count')),
                                                                 ('scaler',
                                                                  StandardScaler())]))])),
                ('classifier', RandomForestClassifier())])

# **Step 15 : Predictions**

In [35]:
predictions = model.predict(X_test)

# **Step 16 : Evaluate Model**

In [36]:
print("Accuracy:", accuracy_score(y_test, predictions))

print(classification_report(y_test, predictions))

Accuracy: 1.0
              precision    recall  f1-score   support

           1       1.00      1.00      1.00       214

    accuracy                           1.00       214
   macro avg       1.00      1.00      1.00       214
weighted avg       1.00      1.00      1.00       214



# **Step 17 : Hyperparameter Tuning**

In [37]:
from sklearn.model_selection import GridSearchCV

params = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20]
}

grid = GridSearchCV(model, params, cv=3)

grid.fit(X_train, y_train)

print(grid.best_params_)

{'classifier__max_depth': 10, 'classifier__n_estimators': 100}


# **Expected Results**

**You may achieve:**

| Metric    | Score    |
| --------- | -------- |
| Accuracy  | 80%–92%  |
| Precision | High     |
| Recall    | Balanced |
| F1 Score  | Strong   |

depending on dataset quality.

# **What You Learned from This Project**

**Key Learnings**

>How to work with mixed-format datasets

>Combining NLP + numerical features

>Building reusable ML pipelines

>Text vectorization using TF-IDF

>Predicting popularity using machine learning

>Hyperparameter tuning

>Real-world recommendation system concepts

# **Real-World Applications**

**This project can be used in:**

Amazon Kindle

Goodreads

Online bookstores

Recommendation engines

Bestseller prediction systems

**In this project, I built a Machine Learning model that predicts book popularity using both textual and numerical features. I applied NLP techniques such as TF-IDF vectorization to process book titles and descriptions and combined them with rating-based features to train a predictive model. This project helped me understand feature engineering, NLP pipelines, data preprocessing, and machine learning model optimization in a real-world recommendation system scenario.**